# Implementasi K-Means untuk Pengelompokan Kinerja Petugas PPL SE2026 (Revisi)
**Studi kasus:** BPS Kabupaten Indragiri Hilir (data SIPANTAU)

Cara pakai di Google Colab:
1. Jalankan sel dari atas ke bawah (**Runtime → Run all**).
2. Saat diminta, unggah file `laporan_progress_petugas_....csv` (data PPL).
3. Hasil gambar, tabel, dan file CSV tersimpan di folder `hasil/` dan otomatis diunduh di sel terakhir.

Parameter yang dapat diubah ada di sel **2. Parameter**.

**Catatan revisi (penting).** Pada versi sebelumnya, kolom tanggal (dd/mm) dianggap sebagai jumlah laporan *per hari*.
Setelah diperiksa, kolom tersebut ternyata berisi **jumlah laporan kumulatif** pada hari petugas memperbarui laporannya
(hari tanpa pembaruan bernilai 0): nilai `TOTAL` selalu sama dengan nilai terbesar di baris tersebut, dan nilai tidak-nol
tidak pernah turun. Karena itu pada versi ini kolom tanggal **diubah dahulu menjadi selisih (laporan baru per hari pelaporan)**
sebelum atribut `rata_harian` dan `volatilitas` dihitung (lihat sel 4 dan 5).

## 1. Import Pustaka

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, adjusted_rand_score)

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
os.makedirs('hasil', exist_ok=True)

## 2. Parameter

In [ ]:
K_MIN, K_MAX = 2, 7      # rentang jumlah klaster yang diuji
K_FINAL      = 3         # jumlah klaster yang dipakai (ubah setelah melihat grafik evaluasi)
RANDOM_STATE = 42
N_INIT       = 10
BATAS_RASIO  = 1.0       # rasio penyelesaian dibatasi maksimum 100%
N_SEED_UJI   = 20        # jumlah seed untuk uji kestabilan klaster

## 3. Unggah dan Baca Data

In [ ]:
# Di Google Colab: muncul tombol unggah. Di komputer lokal: isi CSV_PATH.
try:
    from google.colab import files
    unggahan = files.upload()
    NAMA_FILE = list(unggahan.keys())[0]
except ImportError:
    NAMA_FILE = os.environ.get('CSV_PATH', 'laporan_progress_petugas.csv')

raw = pd.read_csv(NAMA_FILE)
kolom_tanggal = [c for c in raw.columns if '/' in c]      # kolom tanggal dd/mm
print('Berkas          :', NAMA_FILE)
print('Jumlah petugas  :', len(raw))
print('Jumlah hari     :', len(kolom_tanggal), f'({kolom_tanggal[0]} s.d. {kolom_tanggal[-1]})')
raw.iloc[:5, :8]

## 4. Pemeriksaan Kualitas Data dan Verifikasi Sifat Kumulatif

Langkah ini memeriksa kualitas data dasar, lalu **memverifikasi apakah kolom tanggal berisi nilai harian atau kumulatif**,
dan mengubahnya menjadi laporan baru per hari (`harian`) jika kumulatif.

In [ ]:
kum = raw[kolom_tanggal].astype(float)       # isi kolom tanggal apa adanya

print('Nilai kosong             :', int(raw.isna().sum().sum()))
print('Nilai negatif (tanggal)  :', int((kum < 0).sum().sum()))
print('Target = 0               :', int((raw['Target'] == 0).sum()))

nama_norm = raw['Nama Petugas'].str.lower().str.strip()
print('Baris dengan nama ganda  :', int(nama_norm.duplicated(keep=False).sum()),
      f'({nama_norm[nama_norm.duplicated(keep=False)].nunique()} nama)')

print('\nStatus penyelesaian:')
print(raw['Status'].value_counts())

In [ ]:
# --- Uji sifat kumulatif kolom tanggal ---
nilai_maks   = kum.max(axis=1)
nilai_terakhir = kum.apply(lambda r: r[r > 0].iloc[-1] if (r > 0).any() else 0.0, axis=1)

def tidak_turun(r):
    v = r[r > 0].values
    return len(v) < 2 or bool(np.all(np.diff(v) >= 0))

print('TOTAL == nilai maksimum baris        :', int((raw['TOTAL'] == nilai_maks).sum()), 'dari', len(raw))
print('TOTAL == nilai tidak-nol terakhir    :', int((raw['TOTAL'] == nilai_terakhir).sum()), 'dari', len(raw))
print('Baris dengan nilai tidak-nol tak turun:', int(kum.apply(tidak_turun, axis=1).sum()), 'dari', len(raw))
print()
print('Jumlah seluruh kolom tanggal (jika salah dianggap harian):', int(kum.sum().sum()))
print('Jumlah TOTAL                                            :', int(raw['TOTAL'].sum()))
print('Petugas dengan TOTAL != jumlah kolom tanggal             :', int((raw['TOTAL'] != kum.sum(axis=1)).sum()))

In [ ]:
# --- Ubah kumulatif -> laporan baru per hari pelaporan ---
# 1) hari tanpa pembaruan (0) diisi nilai kumulatif sebelumnya, 2) diambil selisih antarhari.
kum_isi = kum.replace(0, np.nan).ffill(axis=1).fillna(0)
harian  = kum_isi.diff(axis=1)
harian.iloc[:, 0] = kum_isi.iloc[:, 0]

assert (harian >= 0).all().all(),               'ada selisih negatif'
assert (harian.sum(axis=1) == raw['TOTAL']).all(), 'jumlah selisih harus sama dengan TOTAL'
print('Verifikasi lolos: jumlah laporan baru per hari = TOTAL untuk seluruh', len(raw), 'petugas.')
print('Total laporan baru seluruh petugas:', int(harian.sum().sum()))

## 5. Pembentukan Atribut Kinerja

Atribut dihitung dari `harian` (laporan baru per hari), **bukan** dari nilai kumulatif mentah.

In [ ]:
df = pd.DataFrame({
    'Nama Petugas': raw['Nama Petugas'],
    'TOTAL': raw['TOTAL'],
    'Target': raw['Target'],
    'Status': raw['Status'],
})
n_hari = len(kolom_tanggal)

df['rasio_penyelesaian'] = (raw['TOTAL'] / raw['Target']).clip(upper=BATAS_RASIO)
df['rata_harian']        = harian.sum(axis=1) / n_hari          # = TOTAL / jumlah hari
df['hari_aktif']         = (harian > 0).sum(axis=1)
df['persen_hari_aktif']  = df['hari_aktif'] / n_hari * 100
df['volatilitas']        = harian.std(axis=1)                   # simpangan baku laporan baru per hari

ATRIBUT = ['rasio_penyelesaian', 'rata_harian', 'persen_hari_aktif', 'volatilitas']
print('Petugas tanpa laporan sama sekali:', int((df['hari_aktif'] == 0).sum()))
print('Petugas yang pernah melapor      :', int((df['hari_aktif'] > 0).sum()))
df[ATRIBUT].describe().round(3)

## 6. Pemisahan Petugas Tanpa Laporan

In [ ]:
tidak_aktif = df[df['hari_aktif'] == 0].copy()
aktif       = df[df['hari_aktif'] > 0].copy().reset_index(drop=True)

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.bar(['Pernah melapor', 'Tanpa laporan'], [len(aktif), len(tidak_aktif)],
       color=['#2a6f97', '#adb5bd'])
for i, v in enumerate([len(aktif), len(tidak_aktif)]):
    ax.text(i, v + 5, f'{v} ({v/len(df)*100:.1f}%)', ha='center')
ax.set_ylabel('Jumlah petugas'); ax.set_title('Keaktifan Pelaporan Petugas PPL')
ax.set_ylim(0, max(len(aktif), len(tidak_aktif)) * 1.15)
plt.tight_layout(); plt.savefig('hasil/gambar_4_1_keaktifan.png', dpi=200); plt.show()

## 7. Korelasi Antaratribut, Transformasi Logaritma, dan Normalisasi Min-Max

Rasio penyelesaian dan rata-rata harian pada dasarnya sama-sama mengukur volume laporan (`TOTAL`), sehingga korelasinya
perlu dilaporkan.

In [ ]:
data_olah = aktif[ATRIBUT].copy()
data_olah['rata_harian'] = np.log1p(data_olah['rata_harian'])   # tangani sebaran condong
data_olah['volatilitas'] = np.log1p(data_olah['volatilitas'])

print('Korelasi Pearson antaratribut (setelah log, sebelum normalisasi):')
korelasi = data_olah.corr().round(3)
korelasi.to_csv('hasil/korelasi_atribut.csv')
print(korelasi)

scaler = MinMaxScaler()
Z = scaler.fit_transform(data_olah)
Z_df = pd.DataFrame(Z, columns=ATRIBUT)
Z_df.describe().round(3)

## 8. Menentukan Jumlah Klaster (Elbow, Silhouette, Davies-Bouldin)

In [ ]:
hasil_k = []
for k in range(K_MIN, K_MAX + 1):
    km = KMeans(n_clusters=k, init='k-means++', n_init=N_INIT, random_state=RANDOM_STATE).fit(Z)
    hasil_k.append({
        'k': k,
        'WCSS': km.inertia_,
        'Silhouette': silhouette_score(Z, km.labels_),
        'Davies-Bouldin': davies_bouldin_score(Z, km.labels_),
        'Ukuran klaster': sorted(np.bincount(km.labels_).tolist(), reverse=True),
    })
evaluasi = pd.DataFrame(hasil_k)
evaluasi['Penurunan WCSS'] = -evaluasi['WCSS'].diff()
evaluasi.round(3)

In [ ]:
# Gambar 4.2 Elbow
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(evaluasi['k'], evaluasi['WCSS'], marker='o', color='#2a6f97')
ax.axvline(K_FINAL, ls='--', color='#c1121f', label=f'k = {K_FINAL}')
ax.set_xlabel('Jumlah klaster (k)'); ax.set_ylabel('WCSS')
ax.set_title('Elbow Method'); ax.legend()
plt.tight_layout(); plt.savefig('hasil/gambar_4_2_elbow.png', dpi=200); plt.show()

# Gambar 4.3 Silhouette dan Davies-Bouldin
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
axs[0].plot(evaluasi['k'], evaluasi['Silhouette'], marker='o', color='#2a9d8f')
axs[0].axvline(K_FINAL, ls='--', color='#c1121f')
axs[0].set_title('Silhouette Score (makin besar makin baik)')
axs[1].plot(evaluasi['k'], evaluasi['Davies-Bouldin'], marker='o', color='#e76f51')
axs[1].axvline(K_FINAL, ls='--', color='#c1121f')
axs[1].set_title('Davies-Bouldin Index (makin kecil makin baik)')
for a in axs: a.set_xlabel('Jumlah klaster (k)')
plt.tight_layout(); plt.savefig('hasil/gambar_4_3_silhouette_dbi.png', dpi=200); plt.show()

## 9. Klasterisasi K-Means dengan k Terpilih

In [ ]:
kmeans = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=N_INIT, random_state=RANDOM_STATE).fit(Z)
aktif['klaster_mentah'] = kmeans.labels_

# Urutkan klaster dari kinerja tertinggi ke terendah berdasarkan rata-rata rasio penyelesaian
urutan = (aktif.groupby('klaster_mentah')['rasio_penyelesaian'].mean()
                .sort_values(ascending=False).index.tolist())
peta = {lama: baru + 1 for baru, lama in enumerate(urutan)}
aktif['klaster'] = aktif['klaster_mentah'].map(peta)

nama_tingkat = {1: 'Tinggi', 2: 'Sedang', 3: 'Rendah'} if K_FINAL == 3 else {i: f'Klaster {i}' for i in range(1, K_FINAL + 1)}
aktif['kelompok'] = aktif['klaster'].map(lambda c: f'Klaster {c} ({nama_tingkat[c]})')

sil = silhouette_score(Z, kmeans.labels_)
dbi = davies_bouldin_score(Z, kmeans.labels_)
print(f'k = {K_FINAL} | Silhouette = {sil:.3f} | Davies-Bouldin = {dbi:.3f} | WCSS = {kmeans.inertia_:.3f}')
print(aktif['kelompok'].value_counts().sort_index())

In [ ]:
# Uji kestabilan: ulangi K-Means dengan banyak seed, bandingkan dengan hasil utama (ARI 1 = identik)
ari, wcss = [], []
for s in range(N_SEED_UJI):
    km_s = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=N_INIT, random_state=s).fit(Z)
    ari.append(adjusted_rand_score(kmeans.labels_, km_s.labels_)); wcss.append(km_s.inertia_)
print(f'Kestabilan {N_SEED_UJI} seed: ARI min = {min(ari):.3f}, rata-rata = {np.mean(ari):.3f} | '
      f'WCSS min-maks = {min(wcss):.3f} - {max(wcss):.3f}')

## 10. Profil Klaster

In [ ]:
profil = (aktif.groupby('kelompok')
          .agg(jumlah_petugas=('Nama Petugas', 'count'),
               rasio_penyelesaian=('rasio_penyelesaian', 'mean'),
               rata_harian=('rata_harian', 'mean'),
               persen_hari_aktif=('persen_hari_aktif', 'mean'),
               volatilitas=('volatilitas', 'mean'),
               hari_aktif=('hari_aktif', 'mean'),
               total_diterima=('TOTAL', 'sum'))
          .round(3))
profil['porsi_total_diterima_%'] = (profil['total_diterima'] / aktif['TOTAL'].sum() * 100).round(1)
profil

In [ ]:
# Sebaran status penyelesaian (Complete/Incomplete) pada setiap klaster
print(pd.crosstab(aktif['kelompok'], aktif['Status']))
print()
print('Petugas tanpa laporan:', tidak_aktif['Status'].value_counts().to_dict())

# Silhouette per klaster
aktif['silhouette'] = silhouette_samples(Z, kmeans.labels_)
print()
print(aktif.groupby('kelompok')['silhouette'].mean().round(3))

In [ ]:
# Gambar 4.4 Sebaran klaster (proyeksi PCA 2 dimensi)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
P = pca.fit_transform(Z)
warna = {1: '#2a9d8f', 2: '#e9c46a', 3: '#e76f51'}
fig, ax = plt.subplots(figsize=(6.5, 4.6))
for c in sorted(aktif['klaster'].unique()):
    m = (aktif['klaster'] == c).values
    ax.scatter(P[m, 0], P[m, 1], s=22, alpha=0.8, color=warna.get(c, None),
               label=f"{aktif.loc[m, 'kelompok'].iloc[0]} (n={m.sum()})")
pusat = pca.transform(kmeans.cluster_centers_)
ax.scatter(pusat[:, 0], pusat[:, 1], marker='X', s=160, c='black', label='Centroid')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varians)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varians)')
ax.set_title('Sebaran Klaster Petugas PPL (Proyeksi PCA)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('hasil/gambar_4_4_sebaran_klaster.png', dpi=200); plt.show()
print('Varians dijelaskan PC1, PC2 (%):', (pca.explained_variance_ratio_ * 100).round(1),
      '| total', round(pca.explained_variance_ratio_.sum() * 100, 1))

# Gambar 4.5 Profil klaster (rata-rata atribut ternormalisasi)
Z_df['klaster'] = aktif['klaster'].values
prof_norm = Z_df.groupby('klaster')[ATRIBUT].mean()
label_atribut = ['Rasio penyelesaian', 'Rata-rata harian (log)', '% hari aktif', 'Volatilitas (log)']
fig, ax = plt.subplots(figsize=(8, 4))
lebar = 0.8 / len(prof_norm)
for i, (c, baris) in enumerate(prof_norm.iterrows()):
    ax.bar(np.arange(4) + i * lebar, baris.values, lebar, color=warna.get(c, None),
           label=f'Klaster {c} ({nama_tingkat[c]})')
ax.set_xticks(np.arange(4) + lebar * (len(prof_norm) - 1) / 2); ax.set_xticklabels(label_atribut)
ax.set_ylabel('Rata-rata nilai ternormalisasi (0-1)'); ax.set_title('Profil Klaster Berdasarkan Atribut'); ax.legend()
plt.tight_layout(); plt.savefig('hasil/gambar_4_5_profil_klaster.png', dpi=200); plt.show()
Z_df.drop(columns='klaster', inplace=True)

## 11. Simpan Hasil

In [ ]:
hasil = pd.concat([
    aktif[['Nama Petugas', 'TOTAL', 'Target', 'Status'] + ATRIBUT + ['hari_aktif', 'kelompok']],
    tidak_aktif[['Nama Petugas', 'TOTAL', 'Target', 'Status'] + ATRIBUT + ['hari_aktif']].assign(kelompok='Tanpa laporan'),
], ignore_index=True)
hasil.to_csv('hasil/hasil_klasterisasi_ppl.csv', index=False, encoding='utf-8-sig')
evaluasi.to_csv('hasil/evaluasi_jumlah_klaster.csv', index=False)
profil.to_csv('hasil/profil_klaster.csv')
print(hasil['kelompok'].value_counts())

# Unduh semua hasil (hanya di Google Colab)
try:
    import shutil
    from google.colab import files
    shutil.make_archive('hasil_klasterisasi', 'zip', 'hasil')
    files.download('hasil_klasterisasi.zip')
except ImportError:
    pass